# Marketing Campaign Response Prediction

This notebook builds a machine learning pipeline to predict whether a customer will respond to a marketing campaign using the `marketing_campaign.csv` dataset.

Models compared with 5-fold cross-validation:
- Logistic Regression
- Random Forest
- Gradient Boosting
- Support Vector Machine (SVM)

## 1. Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import make_scorer, precision_score, recall_score, f1_score, roc_curve, precision_recall_curve
from sklearn.model_selection import StratifiedKFold, cross_validate, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

print("All libraries loaded.")

## 2. Load the Dataset

In [ ]:
df = pd.read_csv('marketing_campaign.csv', sep='\t')
print('Shape:', df.shape)
df.head()

## 3. Quick Data Check

In [ ]:
print(df.info())
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False).head(10))

print('\nTarget distribution:')
print(df['Response'].value_counts())
print(df['Response'].value_counts(normalize=True).rename('proportion'))

## 4. Feature Engineering

In [ ]:
data = df.copy()

# Customer tenure
data['Dt_Customer'] = pd.to_datetime(data['Dt_Customer'], format='%d-%m-%Y', errors='coerce')
latest_date = data['Dt_Customer'].max()
data['Customer_Days'] = (latest_date - data['Dt_Customer']).dt.days

# Age
data['Age'] = 2026 - data['Year_Birth']

# Children & spending
data['Total_Children'] = data['Kidhome'] + data['Teenhome']
spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
data['Total_Spending'] = data[spend_cols].sum(axis=1)

# Campaign history — strong predictor
cmp_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']
data['Total_Accepted'] = data[cmp_cols].sum(axis=1)

# Financial ratios
data['Income_Per_Person'] = data['Income'] / (1 + data['Total_Children'])
data['Spending_Rate'] = data['Total_Spending'] / (data['Income'] + 1)

# Web/store engagement ratio
data['Web_Purchase_Rate'] = data['NumWebPurchases'] / (data['NumWebVisitsMonth'] + 1)

# Drop identifiers and raw date
data = data.drop(columns=['ID', 'Dt_Customer', 'Year_Birth'])

print(f"Features after engineering: {data.shape[1] - 1}")
data.head()

## 5. Prepare Features and Target

In [ ]:
X = data.drop(columns=['Response'])
y = data['Response']

categorical_features = ['Education', 'Marital_Status']
numeric_features = [col for col in X.columns if col not in categorical_features]

print('Number of features before encoding:', X.shape[1])
print('Numeric features:', len(numeric_features))
print('Categorical features:', categorical_features)

## 6. Build Preprocessing Pipelines

In [ ]:
numeric_transformer_scaled = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

numeric_transformer_unscaled = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_scaled = ColumnTransformer(transformers=[
    ('num', numeric_transformer_scaled, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

preprocessor_unscaled = ColumnTransformer(transformers=[
    ('num', numeric_transformer_unscaled, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

## 7. Define Models

In [ ]:
smote = SMOTE(random_state=42)

models = {
    'Logistic Regression': Pipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('model', LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42))
    ]),
    'Random Forest': ImbPipeline(steps=[
        ('preprocessor', preprocessor_unscaled),
        ('sampler', SMOTE(random_state=42)),
        ('model', RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42))
    ]),
    'Gradient Boosting': ImbPipeline(steps=[
        ('preprocessor', preprocessor_unscaled),
        ('sampler', SMOTE(random_state=42)),
        ('model', GradientBoostingClassifier(random_state=42))
    ]),
    'SVM': Pipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('model', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))
    ]),
    'XGBoost': ImbPipeline(steps=[
        ('preprocessor', preprocessor_unscaled),
        ('sampler', SMOTE(random_state=42)),
        ('model', XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                use_label_encoder=False, eval_metric='logloss',
                                random_state=42, n_jobs=-1))
    ]),
    'LightGBM': ImbPipeline(steps=[
        ('preprocessor', preprocessor_unscaled),
        ('sampler', SMOTE(random_state=42)),
        ('model', LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=4,
                                  class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1))
    ]),
}

list(models.keys())

## 8. Run 5-Fold Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    'accuracy': 'accuracy',
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0),
    'roc_auc': 'roc_auc'
}

results = []

for model_name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=1)
    results.append({
        'Model': model_name,
        'Accuracy Mean': scores['test_accuracy'].mean(),
        'Accuracy Std': scores['test_accuracy'].std(),
        'Precision Mean': scores['test_precision'].mean(),
        'Recall Mean': scores['test_recall'].mean(),
        'F1 Mean': scores['test_f1'].mean(),
        'ROC-AUC Mean': scores['test_roc_auc'].mean()
    })

results_df = pd.DataFrame(results).sort_values(by='ROC-AUC Mean', ascending=False)
results_df.reset_index(drop=True)

## 9. Visualize Model Performance

In [ ]:
plot_df = results_df[['Model', 'Accuracy Mean', 'F1 Mean', 'ROC-AUC Mean']].set_index('Model')

ax = plot_df.plot(kind='bar', figsize=(10, 6))
ax.set_title('Model Comparison Using 5-Fold Cross-Validation')
ax.set_ylabel('Score')
ax.set_xlabel('Model')
plt.xticks(rotation=20)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 10. Conclusion

In [ ]:
best_model = results_df.iloc[0]
print('Best model based on ROC-AUC:')
print(best_model)

print('\nInterpretation:')
print('This dataset is well suited for a supervised binary classification task where the target is customer response to a marketing campaign.')

## 11. Hyperparameter Tuning (Best Model)

In [ ]:
# Tune LightGBM (fastest and typically best performer)
lgbm_pipe = ImbPipeline(steps=[
    ('preprocessor', preprocessor_unscaled),
    ('sampler', SMOTE(random_state=42)),
    ('model', LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1))
])

param_dist = {
    'model__n_estimators': [200, 300, 500],
    'model__learning_rate': [0.01, 0.05, 0.1],
    'model__max_depth': [3, 4, 6, -1],
    'model__num_leaves': [31, 63, 127],
    'model__min_child_samples': [10, 20, 30],
    'model__subsample': [0.7, 0.8, 1.0],
    'model__colsample_bytree': [0.7, 0.8, 1.0],
}

search = RandomizedSearchCV(
    lgbm_pipe,
    param_distributions=param_dist,
    n_iter=30,
    scoring='roc_auc',
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    random_state=42,
    n_jobs=1,
    verbose=1
)
search.fit(X, y)

print(f"\nBest ROC-AUC (CV): {search.best_score_:.4f}")
print("Best params:", search.best_params_)

## 12. Decision Threshold Optimization

In [ ]:
from sklearn.model_selection import train_test_split

# Use best model from tuning
best_model = search.best_estimator_

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
best_model.fit(X_train, y_train)
y_proba = best_model.predict_proba(X_val)[:, 1]

# Find threshold maximizing F1
precisions, recalls, thresholds = precision_recall_curve(y_val, y_proba)
f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
best_idx = f1_scores[:-1].argmax()
best_threshold = thresholds[best_idx]

y_pred_tuned = (y_proba >= best_threshold).astype(int)

print(f"Optimal threshold: {best_threshold:.3f}")
print(f"F1  @ optimal threshold: {f1_score(y_val, y_pred_tuned):.4f}")
print(f"Precision:                {precision_score(y_val, y_pred_tuned):.4f}")
print(f"Recall:                   {recall_score(y_val, y_pred_tuned):.4f}")

# Plot PR curve
plt.figure(figsize=(8, 5))
plt.plot(recalls[:-1], precisions[:-1], label='PR Curve')
plt.scatter(recalls[best_idx], precisions[best_idx], color='red', zorder=5,
            label=f'Best threshold = {best_threshold:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve with Optimal Threshold')
plt.legend()
plt.tight_layout()
plt.show()